# Voice Live and voice agents — companion notebook

Companion to the deck **Voice Live and voice agents** (October 2026). Each step names the slide it
illustrates.

**The scenario throughout: an IT service desk phone line.** Employees call to ask about a support
ticket or a common problem such as a password reset. The goal is a voice assistant that answers
simple questions, looks up ticket status in a back-end system, and hands everything else to a person.
Each step adds one capability on the way there.

**The steps get harder on purpose.**

| Part | Steps | What it shows | Status | Deck |
|---|---|---|---|---|
| A · Voice Live API | 1–5 | First session; voices; listening and turn detection; model choice; tool calls | generally available | slides 2, 4, 5 |
| B · Voice Live with a Foundry agent | 6 | Give an existing text agent a voice, by agent name | generally available | slides 3, 7 |
| C · Foundry voice agent | 7–9 | Define a voice agent; talk to it; read the conversation back | **preview** | slides 6, 8 |

**How this notebook handles audio.** A live microphone loop doesn't work well inside Jupyter, so
the notebook types the caller's words and **plays the agent's spoken reply in the notebook**. For
real speech input, step 2 generates a short recording of a "caller", and step 3 streams it into
the session exactly as a microphone would. No microphone, speakers driver or PortAudio
installation is needed.

**Preview means preview.** Foundry voice agents (part C) are in public preview, with **no service
level agreement**. The preview must be available in your subscription and region, and its API names
can change before general availability.

**Docs**
- Voice Live overview — <https://learn.microsoft.com/en-us/azure/ai-services/speech-service/voice-live>
- Voice Live how-to — <https://learn.microsoft.com/en-us/azure/ai-services/speech-service/voice-live-how-to>
- Voice Live with Foundry agents — <https://learn.microsoft.com/en-us/azure/ai-services/speech-service/how-to-voice-agent-integration>
- Configure a voice agent — <https://learn.microsoft.com/en-us/azure/foundry/agents/how-to/configure-voice-agent>

## 0 · Prerequisites and setup

**On the Azure side** (about 10 minutes):

1. A **Microsoft Foundry resource and project** in a region that supports Voice Live with agents.
   Examples include Sweden Central, West Europe and Switzerland North; check the
   [region table](https://learn.microsoft.com/en-us/azure/ai-services/speech-service/regions) first.
2. **Roles for your own account** on the resource: *Cognitive Services User* (for Voice Live) and
   *Foundry User* (to create and call agents). Role assignments can take a few minutes to apply.
3. **One text model deployment** in the project, for example `gpt-4.1-mini`. Only step 6 needs it,
   because it creates an ordinary text agent.
4. **Part C only:** access to the voice agents preview in your subscription and region.

**Models for Voice Live itself (steps 1–5) need no deployment.** Voice Live is fully managed, and
the listed models are available by name.

**On your machine:** Python 3.10 or later, and the Azure CLI. Sign in with `az login`. The notebook
uses **Microsoft Entra ID** throughout; API keys are not used, because agent mode doesn't support
them.

Copy `.env.example` to `.env` in the repository root and fill in the Voice Live section:

```
AZURE_VOICELIVE_ENDPOINT=https://<your-resource>.services.ai.azure.com/
FOUNDRY_PROJECT_ENDPOINT=https://<your-resource>.services.ai.azure.com/api/projects/<your-project>
FOUNDRY_PROJECT_NAME=<your-project>
FOUNDRY_TEXT_MODEL_DEPLOYMENT=<your-text-model-deployment>     # step 6 only
VOICELIVE_REALTIME_MODEL=gpt-realtime                           # optional
VOICELIVE_TEXT_MODEL=gpt-4.1-mini                               # optional
VOICE_AGENT_MODEL=gpt-realtime                                  # optional, part C
```

In [ ]:
%pip install -q -r requirements.txt

In [ ]:
import asyncio, base64, io, json, os, time, wave
from dotenv import load_dotenv
from IPython.display import Audio, display

from azure.identity.aio import DefaultAzureCredential as AsyncCredential
from azure.ai.voicelive.aio import connect
from azure.ai.voicelive.models import (
    AudioEchoCancellation,
    AudioInputTranscriptionOptions,
    AudioNoiseReduction,
    AzureStandardVoice,
    FunctionCallOutputItem,
    FunctionTool,
    InputAudioFormat,
    InputTextContentPart,
    ItemType,
    MessageItem,
    Modality,
    OutputAudioFormat,
    RequestSession,
    ServerEventType,
    ServerVad,
    ToolChoiceLiteral,
)

load_dotenv()

VL_ENDPOINT      = os.getenv("AZURE_VOICELIVE_ENDPOINT", "")
PROJECT_ENDPOINT = os.getenv("FOUNDRY_PROJECT_ENDPOINT", "")
PROJECT_NAME     = os.getenv("FOUNDRY_PROJECT_NAME", "")
TEXT_DEPLOYMENT  = os.getenv("FOUNDRY_TEXT_MODEL_DEPLOYMENT", "")
REALTIME_MODEL   = os.getenv("VOICELIVE_REALTIME_MODEL", "gpt-realtime")
TEXT_MODEL       = os.getenv("VOICELIVE_TEXT_MODEL", "gpt-4.1-mini")
VOICE_AGENT_MODEL = os.getenv("VOICE_AGENT_MODEL", "gpt-realtime")

API_VERSION = "2026-07-15"     # generally available; pin it rather than relying on the SDK default
RATE        = 24000            # audio format used throughout: PCM16, mono, 24 kHz

for name, value in [("AZURE_VOICELIVE_ENDPOINT", VL_ENDPOINT),
                    ("FOUNDRY_PROJECT_ENDPOINT", PROJECT_ENDPOINT),
                    ("FOUNDRY_PROJECT_NAME", PROJECT_NAME),
                    ("FOUNDRY_TEXT_MODEL_DEPLOYMENT", TEXT_DEPLOYMENT)]:
    print(f"{name:32s} {'set' if value else 'MISSING'}")

### The three helpers everything else uses

A Voice Live session is one WebSocket connection. Both sides exchange **events**, and every step
below follows the same rhythm:

1. send `session.update` to configure the session, and wait for `session.updated`;
2. add something to the conversation (typed text, audio or a tool result) and ask for a response;
3. read events until `response.done`, keeping the audio and the transcripts.

`collect()` does step 3 and records **when the first audio arrived**, which is the latency the caller
notices. `say()` does steps 2 and 3 for a typed message. `play()` turns the collected PCM bytes into
a WAV file the notebook can play.

In [ ]:
def to_wav(pcm: bytes, rate: int = RATE) -> bytes:
    """Wrap raw PCM16 mono audio in a WAV container."""
    buf = io.BytesIO()
    with wave.open(buf, "wb") as w:
        w.setnchannels(1); w.setsampwidth(2); w.setframerate(rate)
        w.writeframes(pcm)
    return buf.getvalue()


def play(rec, label=""):
    """Show the transcript and play the spoken reply."""
    secs = len(rec["audio"]) / 2 / RATE
    first = f"{rec['first_audio_s']:.2f}s" if rec["first_audio_s"] is not None else "n/a"
    print(f"{label}  audio {secs:.1f}s  |  first audio after {first}")
    if rec["user_transcript"]:
        print("  caller :", rec["user_transcript"])
    print("  agent  :", rec["transcript"] or "(no transcript)")
    if rec["audio"]:
        display(Audio(to_wav(bytes(rec["audio"])), rate=RATE))


async def collect(conn, until=ServerEventType.RESPONSE_DONE, timeout=60, verbose=False):
    """Read events until `until` arrives. Returns audio, transcripts, tool calls and timings."""
    rec = {"audio": bytearray(), "transcript": "", "user_transcript": "",
           "events": [], "first_audio_s": None, "calls": [], "session": None}
    t0 = time.perf_counter()

    async def _loop():
        pending_names = {}
        async for ev in conn:
            rec["events"].append(ev.type)
            if verbose:
                print(f"  {time.perf_counter() - t0:6.2f}s  {ev.type}")
            if ev.type in (ServerEventType.SESSION_CREATED, ServerEventType.SESSION_UPDATED):
                rec["session"] = ev.session
            elif ev.type == ServerEventType.RESPONSE_AUDIO_DELTA:
                if rec["first_audio_s"] is None:
                    rec["first_audio_s"] = time.perf_counter() - t0
                rec["audio"] += ev.delta
            elif ev.type == ServerEventType.RESPONSE_AUDIO_TRANSCRIPT_DONE:
                rec["transcript"] += ev.get("transcript", "")
            elif ev.type == ServerEventType.CONVERSATION_ITEM_INPUT_AUDIO_TRANSCRIPTION_COMPLETED:
                rec["user_transcript"] = ev.get("transcript", "")
            elif ev.type == ServerEventType.CONVERSATION_ITEM_CREATED and ev.item.type == ItemType.FUNCTION_CALL:
                pending_names[ev.item.call_id] = (ev.item.name, ev.item.id)
            elif ev.type == ServerEventType.RESPONSE_FUNCTION_CALL_ARGUMENTS_DONE:
                name, item_id = pending_names.get(ev.call_id, (ev.get("name"), None))
                rec["calls"].append({"name": name, "call_id": ev.call_id,
                                     "arguments": ev.arguments, "item_id": item_id})
            elif ev.type == ServerEventType.ERROR:
                raise RuntimeError(ev.error.message)
            if ev.type == until:
                return

    await asyncio.wait_for(_loop(), timeout)
    return rec


async def say(conn, text, **kw):
    """Add a typed user message to the conversation, ask for a response, and collect it."""
    await conn.conversation.item.create(
        item=MessageItem(role="user", content=[InputTextContentPart(text=text)]))
    await conn.response.create()
    return await collect(conn, **kw)


credential = AsyncCredential()   # Microsoft Entra ID via your `az login` session

---
# Part A · The Voice Live API on its own

In part A, **your application owns the conversation logic**: it sends the instructions and the
tools in `session.update`. This is path 1 on deck slide 3, and the closest match to the Azure OpenAI
Realtime API.

## 1 · First session: a greeting and one question  *(deck slides 2 and 4)*

The smallest useful session:

- **connect** to the resource with a model name (`gpt-realtime`);
- **configure** it with instructions, a voice and the audio formats;
- ask for a response with **no user input** at all. This is a *proactive greeting*, the agent
  speaking first, as a phone line should;
- then send one **typed** question and listen to the answer.

Notice what is missing: no speech-to-text service, no text-to-speech service and no orchestration
code. The single WebSocket session handles all of it.

In [ ]:
DESK_INSTRUCTIONS = """
You are the voice assistant for the Contoso IT service desk phone line.
Always speak English, unless the caller speaks another language; then answer in theirs.
Speak in short sentences and ask one question at a time.
You can help with password resets and ticket status questions.
For a password reset, tell the caller to use the self-service portal at aka.ms/sspr,
which needs their registered phone. For anything else, offer to transfer to a person.
Never ask for or repeat a password.
""".strip()

session_basic = RequestSession(
    modalities=[Modality.TEXT, Modality.AUDIO],
    instructions=DESK_INSTRUCTIONS,
    voice=AzureStandardVoice(name="en-US-Ava:DragonHDLatestNeural"),
    input_audio_format=InputAudioFormat.PCM16,
    output_audio_format=OutputAudioFormat.PCM16,
)

async with connect(endpoint=VL_ENDPOINT, credential=credential,
                   model=REALTIME_MODEL, api_version=API_VERSION) as conn:
    await conn.session.update(session=session_basic)
    ready = await collect(conn, until=ServerEventType.SESSION_UPDATED)
    print("session id:", ready["session"].id)

    await conn.response.create()                      # agent speaks first
    play(await collect(conn), "greeting")

    play(await say(conn, "Hi, I forgot my password. What should I do?"), "answer")

**What to look at**

- *first audio after …s* is the time from the request to the first audio chunk. It is the delay a
  caller actually hears, and it is usually far shorter than the full reply, because audio is
  **streamed** as `response.audio.delta` events and playback can start straight away.
- The transcript arrives alongside the audio (`response.audio_transcript.done`). Logging and quality
  review use it.
- The event names are the same as in the Azure OpenAI Realtime API. Voice Live is designed to be
  event-compatible with it.

## 2 · The speaking side: voices  *(deck slide 2)*

The voice is a session setting. Here the **same sentence** is spoken by a high-definition (HD) voice
and a standard neural voice, so you can hear the difference. Voice Live offers 600+ standard voices
in 150+ locales; custom voice is available with limited access.

The cell also records a short **"caller" clip** in a third voice and saves it as `caller.wav`.
Step 3 streams that clip into a session as if it came from a microphone. This keeps the notebook
self-contained, with no recording needed.

In [ ]:
LINE = "Your ticket is in progress. An engineer will call you back before five p.m. today."
VOICES = ["en-US-Ava:DragonHDLatestNeural", "en-GB-SoniaNeural"]

READ_ALOUD = "Read the user's text aloud exactly as written. Add nothing before or after it."

for voice in VOICES:
    s = RequestSession(modalities=[Modality.TEXT, Modality.AUDIO], instructions=READ_ALOUD,
                       voice=AzureStandardVoice(name=voice),
                       input_audio_format=InputAudioFormat.PCM16,
                       output_audio_format=OutputAudioFormat.PCM16)
    async with connect(endpoint=VL_ENDPOINT, credential=credential,
                       model=REALTIME_MODEL, api_version=API_VERSION) as conn:
        await conn.session.update(session=s)
        await collect(conn, until=ServerEventType.SESSION_UPDATED)
        play(await say(conn, LINE), voice)

# --- record the simulated caller for step 3 ---
CALLER_LINE = ("Hello, this is Maria from finance. "
               "I'd like to know the status of my ticket, number I N C four seven one one.")
s = RequestSession(modalities=[Modality.TEXT, Modality.AUDIO], instructions=READ_ALOUD,
                   voice=AzureStandardVoice(name="en-US-AndrewNeural"),
                   input_audio_format=InputAudioFormat.PCM16,
                   output_audio_format=OutputAudioFormat.PCM16)
async with connect(endpoint=VL_ENDPOINT, credential=credential,
                   model=REALTIME_MODEL, api_version=API_VERSION) as conn:
    await conn.session.update(session=s)
    await collect(conn, until=ServerEventType.SESSION_UPDATED)
    caller = await say(conn, CALLER_LINE)

with open("caller.wav", "wb") as f:
    f.write(to_wav(bytes(caller["audio"])))
print("saved caller.wav")
display(Audio("caller.wav"))

**Realtime voices vs Azure voices.** With `gpt-realtime` you can also use the model's own built-in
voices, for example `voice="alloy"` as a plain string. Azure voices bring the full voice catalogue,
HD voices and custom voice. In the three-stage setup in step 4, only Azure voices are available.

## 3 · The listening side: streaming audio, turn detection and transcription  *(deck slide 4)*

This is what a phone call really looks like. The client streams **audio**, not text, in small
chunks with `input_audio_buffer.append`. The service then decides **when the caller has finished
speaking**; this is *turn detection*.

This session adds the conversational enhancements from deck slide 2:

| Setting | What it does |
|---|---|
| `turn_detection` (server VAD) | Detects speech start and stop. When the caller stops, the turn ends and a response is created automatically. |
| `input_audio_noise_reduction` | Deep noise suppression before recognition. |
| `input_audio_echo_cancellation` | Stops the agent hearing its own voice on a speakerphone. |
| `input_audio_transcription` | Returns a transcript of what the caller said, for logs and review. |

Watch the event timeline. `speech_started` and `speech_stopped` are turn detection at work, and the
transcription event shows what was heard.

**Note on turn detection types.** This cell uses `ServerVad` (silence-based), which behaves
predictably with a recorded clip. For live callers, Voice Live also offers **Azure semantic voice
activity detection** (`AzureSemanticVad`, `AzureSemanticVadMultilingual`). It uses the *meaning* of
what was said, not only silence, so a caller pausing to think is not cut off. This option is
specific to Voice Live; the Azure OpenAI Realtime API doesn't have it.

In [ ]:
session_listen = RequestSession(
    modalities=[Modality.TEXT, Modality.AUDIO],
    instructions=DESK_INSTRUCTIONS,
    voice=AzureStandardVoice(name="en-US-Ava:DragonHDLatestNeural"),
    input_audio_format=InputAudioFormat.PCM16,
    output_audio_format=OutputAudioFormat.PCM16,
    turn_detection=ServerVad(threshold=0.5, prefix_padding_ms=300, silence_duration_ms=500),
    input_audio_noise_reduction=AudioNoiseReduction(type="azure_deep_noise_suppression"),
    input_audio_echo_cancellation=AudioEchoCancellation(),
    input_audio_transcription=AudioInputTranscriptionOptions(model="whisper-1"),
)

with wave.open("caller.wav", "rb") as w:
    caller_pcm = w.readframes(w.getnframes())

CHUNK = RATE * 2 // 20            # 50 ms of PCM16 audio, the size a microphone callback would send

async def stream_audio(conn, pcm):
    for i in range(0, len(pcm), CHUNK):
        await conn.input_audio_buffer.append(audio=base64.b64encode(pcm[i:i + CHUNK]).decode())
        await asyncio.sleep(0.05)                      # real-time pace, like a live microphone
    silence = bytes(CHUNK)
    for _ in range(30):                                # 1.5 s of silence so the turn can end
        await conn.input_audio_buffer.append(audio=base64.b64encode(silence).decode())
        await asyncio.sleep(0.05)

async with connect(endpoint=VL_ENDPOINT, credential=credential,
                   model=REALTIME_MODEL, api_version=API_VERSION) as conn:
    await conn.session.update(session=session_listen)
    await collect(conn, until=ServerEventType.SESSION_UPDATED)
    print("event timeline:")
    sender = asyncio.create_task(stream_audio(conn, caller_pcm))
    rec = await collect(conn, verbose=True)            # no response.create: turn detection does it
    await sender

play(rec, "reply to the spoken caller")

**What just happened.** No code ever said "the caller is done". The service detected the end of
speech, committed the audio, transcribed it and created a response by itself.

The agent can't actually check the ticket yet, because it has no access to the ticketing system.
Step 5 fixes that.

**Barge-in.** If the caller starts speaking while the agent is talking, the service sends
`input_audio_buffer.speech_started` and interrupts the current response. The client should then
**stop local playback** straight away. This is enabled by default (`interrupt_response`). It is
hard to demonstrate with a pre-recorded clip, so it is described here rather than run.

## 4 · Model choice: single speech model vs three stages  *(deck slide 5)*

The architecture is never set explicitly. **Voice Live derives it from the model name:**

| Model | How audio is processed | Pricing tier |
|---|---|---|
| `gpt-realtime` | **Single speech model**: audio in, audio out | Pro |
| `gpt-4.1-mini` | **Three stages**: Azure speech recognition → text model → Azure speech synthesis | Standard |

**Measure what a caller feels.** The delay that matters on a call runs from the moment the caller
**stops speaking** to the moment the agent **starts speaking**. So the cell streams the step 3
recording (`caller.wav`) into both models with the same turn detection, and times
`input_audio_buffer.speech_stopped` → first `response.audio.delta`. Each model runs three times and
the median is reported, because a single call is dominated by network and service noise.

**Why not typed text?** With typed input the three-stage pipeline skips speech recognition
entirely, so a small text model plus streaming speech synthesis can start speaking *sooner* than
`gpt-realtime`. That measures a case a phone line never has. The single speech model's advantage —
no recognition stage, no hand-offs, and the caller's tone kept rather than reduced to text — only
shows up with spoken input.

Both models also wait for the same `silence_duration_ms` (500 ms) before the turn ends; that wait is
the same for both and comes on top of the numbers below. Your numbers depend on region, load and
network, so treat them as indicative. The three-stage setup still has its own strengths: a wider
choice of text models and voices, phrase lists and custom speech.

In [ ]:
import statistics

RUNS = 3                                    # per model; the median smooths out network noise

session_compare = RequestSession(           # identical for both models, so only the model differs
    modalities=[Modality.TEXT, Modality.AUDIO],
    instructions=DESK_INSTRUCTIONS,
    voice=AzureStandardVoice(name="en-US-Ava:DragonHDLatestNeural"),
    input_audio_format=InputAudioFormat.PCM16,
    output_audio_format=OutputAudioFormat.PCM16,
    turn_detection=ServerVad(threshold=0.5, prefix_padding_ms=300, silence_duration_ms=500),
)

async def spoken_turn(model):
    """Stream caller.wav (from step 3) and time end of speech -> first reply audio."""
    rec = {"audio": bytearray(), "transcript": "", "user_transcript": "", "first_audio_s": None}
    async with connect(endpoint=VL_ENDPOINT, credential=credential,
                       model=model, api_version=API_VERSION) as conn:
        await conn.session.update(session=session_compare)
        await collect(conn, until=ServerEventType.SESSION_UPDATED)
        sender = asyncio.create_task(stream_audio(conn, caller_pcm))
        t_stop = None
        async for ev in conn:
            if ev.type == ServerEventType.INPUT_AUDIO_BUFFER_SPEECH_STOPPED:
                t_stop = time.perf_counter()
            elif ev.type == ServerEventType.RESPONSE_AUDIO_DELTA:
                if rec["first_audio_s"] is None and t_stop is not None:
                    rec["first_audio_s"] = time.perf_counter() - t_stop
                rec["audio"] += ev.delta
            elif ev.type == ServerEventType.RESPONSE_AUDIO_TRANSCRIPT_DONE:
                rec["transcript"] += ev.get("transcript", "")
            elif ev.type == ServerEventType.ERROR:
                raise RuntimeError(ev.error.message)
            elif ev.type == ServerEventType.RESPONSE_DONE:
                break
        await sender
    return rec

timings = {}
for model in [REALTIME_MODEL, TEXT_MODEL]:
    timings[model] = []
    for _ in range(RUNS):
        rec = await asyncio.wait_for(spoken_turn(model), 90)
        timings[model].append(rec["first_audio_s"])
    play(rec, f"{model} (last run)")

print("\nend of speech -> first audio")
for model, runs in timings.items():
    shown = ", ".join(f"{t:.2f}s" for t in runs)
    print(f"{model:16s} median {statistics.median(runs):.2f}s   runs [{shown}]")

## 5 · Calling a back-end system: function tools  *(deck slide 4, step 4)*

Now the assistant gets real data. A **function tool** is declared in the session. When the model
decides it needs it, the service sends the call's **name and arguments**. **Your code** runs the
function (here, a lookup in a small in-memory ticket table standing in for the ticketing system)
and returns the result, and the model then speaks the answer.

The round trip, as events:

```
model decides  →  response.function_call_arguments.done   (name, call_id, arguments)
your code      →  conversation.item.create(function_call_output)  +  response.create
model speaks   →  response.audio.delta …  response.done
```

Two practical points:

- **Tool latency is caller silence.** A slow back end means dead air. Voice Live can fill the gap
  with an *interim response*, a short "let me check that for you", configured with
  `interim_response` on the session.
- **The function runs in your environment**, with your credentials and access rules. Voice Live
  never calls the back end itself.

In [ ]:
TICKETS = {   # stands in for the ticketing system
    "INC4711": {"title": "Laptop docking station not detected", "status": "In progress",
                "assigned_to": "Workplace team", "next_update": "today before 17:00"},
    "INC4802": {"title": "VPN access for new contractor", "status": "Waiting for approval",
                "assigned_to": "Network team", "next_update": "after manager approval"},
}

def get_ticket_status(ticket_id: str) -> dict:
    key = ticket_id.upper().replace("-", "").replace(" ", "")
    return TICKETS.get(key, {"error": f"No ticket found with number {ticket_id}"})

ticket_tool = FunctionTool(
    name="get_ticket_status",
    description="Look up the current status of an IT service desk ticket by its number, for example INC4711.",
    parameters={
        "type": "object",
        "properties": {"ticket_id": {"type": "string", "description": "Ticket number, e.g. INC4711"}},
        "required": ["ticket_id"],
    },
)

session_tools = RequestSession(
    modalities=[Modality.TEXT, Modality.AUDIO],
    instructions=DESK_INSTRUCTIONS + "\nUse get_ticket_status for any question about a ticket.",
    voice=AzureStandardVoice(name="en-US-Ava:DragonHDLatestNeural"),
    input_audio_format=InputAudioFormat.PCM16,
    output_audio_format=OutputAudioFormat.PCM16,
    tools=[ticket_tool],
    tool_choice=ToolChoiceLiteral.AUTO,
)

async with connect(endpoint=VL_ENDPOINT, credential=credential,
                   model=REALTIME_MODEL, api_version=API_VERSION) as conn:
    await conn.session.update(session=session_tools)
    await collect(conn, until=ServerEventType.SESSION_UPDATED)

    first = await say(conn, "What's the status of my ticket INC-4711?")
    for call in first["calls"]:
        args = json.loads(call["arguments"])
        result = get_ticket_status(**args)
        print(f"tool call  {call['name']}({args})  ->  {result}")
        await conn.conversation.item.create(
            item=FunctionCallOutputItem(call_id=call["call_id"], output=json.dumps(result)))
    if first["calls"]:
        await conn.response.create()                   # let the model speak the result
        play(await collect(conn), "answer with ticket data")
    else:
        play(first, "model answered without calling the tool")

**Other tool types.** Besides functions that your code runs, a Voice Live session can use
**remote MCP servers** (Model Context Protocol), which the service calls directly, and built-in
search. The official sample repository has a working MCP quickstart:
<https://github.com/microsoft-foundry/voicelive-samples/tree/main/python/voice-live-quickstarts>.

**End of part A.** Everything so far lives in this notebook's code: instructions, voice, tools.
That is fine for one application, but it means every client must carry the same configuration.
Parts B and C move the logic into **Foundry**.

---
# Part B · Voice Live with an existing Foundry agent  *(deck slides 3 and 7)*

## 6 · Give a text agent a voice, by agent name

Many organisations already run a **text agent** in Foundry Agent Service, with its instructions,
model and tools defined there. Path 2 on deck slide 3 puts Voice Live **in front of that agent**:
the client connects with `agent_name` and `project_name` instead of a model, Voice Live handles
listening and speaking, and the agent does the reasoning. This path is generally available.

The cell first creates a small **prompt agent** for the service desk, so the step is
self-contained. If you already have one, skip creation and set `TEXT_AGENT_NAME` to its name.

Three rules from Microsoft Learn apply in agent mode:

- **Microsoft Entra ID only.** Key-based authentication is not supported.
- **No `instructions` in `session.update`.** The agent owns them. The session can still set the
  voice and audio options.
- **Pin `agent_version`** in production so behaviour doesn't change silently when the agent is
  updated.

In [ ]:
from azure.identity import DefaultAzureCredential as SyncCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition

TEXT_AGENT_NAME = "it-desk-text-agent"

sync_cred = SyncCredential()
project = AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=sync_cred)

text_agent = project.agents.create_version(
    agent_name=TEXT_AGENT_NAME,
    definition=PromptAgentDefinition(model=TEXT_DEPLOYMENT, instructions=DESK_INSTRUCTIONS),
)
print(f"text agent '{TEXT_AGENT_NAME}' version {text_agent.version}")

In [ ]:
session_agent = RequestSession(              # voice and audio only; no instructions in agent mode
    modalities=[Modality.TEXT, Modality.AUDIO],
    voice=AzureStandardVoice(name="en-US-Ava:DragonHDLatestNeural"),
    input_audio_format=InputAudioFormat.PCM16,
    output_audio_format=OutputAudioFormat.PCM16,
)

async with connect(endpoint=VL_ENDPOINT, credential=credential, api_version=API_VERSION,
                   agent_name=TEXT_AGENT_NAME, project_name=PROJECT_NAME,
                   agent_version=str(text_agent.version)) as conn:
    await conn.session.update(session=session_agent)
    ready = await collect(conn, until=ServerEventType.SESSION_UPDATED)
    print("connected to agent:", ready["session"].agent.name)
    play(await say(conn, "Hi, I forgot my password. What should I do?"), "agent answer")

**What changed compared with step 1:** the instructions came from the **agent in Foundry**, not from
this notebook. Update the agent in the Foundry portal, and every voice and text client picks up the
change.

**Known limits of this path** (deck slide 8):

- The agent's own trace covers the **text model only**. Turn detection, audio input and audio output
  are not in it.
- Telephony is **yours to integrate**.
- Only **text models** can sit behind the agent.

Foundry voice agents, in part C, address all three.

In [ ]:
# Clean up the text agent created in this step (skip if you pointed at your own agent)
project.agents.delete_version(agent_name=TEXT_AGENT_NAME, agent_version=text_agent.version)
print("deleted text agent version", text_agent.version)

---
# Part C · Foundry voice agents  *(preview — deck slides 6 and 8)*

A **voice agent** is a new agent *kind* (`kind: voice`) in Foundry Agent Service. One **versioned
definition** holds everything that part A kept in notebook code: model, instructions, audio
settings and tools. It still runs on Voice Live underneath. The voice agent adds built-in telephony
(Teams Phone, Twilio), voice-pipeline tracing and opt-in recording of transcripts and audio.

> **Preview.** No service level agreement, and the API names in the cells below may change. The
> preview must be enabled for your subscription and region. `allow_preview=True` on the client
> opts in to preview operations.

## 7 · Define the service desk voice agent  *(deck slide 6)*

The definition maps one-to-one onto the table on deck slide 6:

| Part | Here |
|---|---|
| Model | `model_type=managed`, `gpt-realtime` (use `self_deployed` for your own deployment) |
| Behaviour | the same service desk instructions |
| Listening | 24 kHz PCM input, server turn detection, transcription |
| Speaking | an Azure HD voice, audio output |
| Actions | the system tool that lets the agent **end the call** politely |
| Data | `store=True`, so the conversation can be read back in step 9 (off by default) |

Every create or update produces a **new immutable version**. The cell creates version 1, then
updates the instructions to show versioning.

In [ ]:
from azure.ai.projects.models import (
    AgentKind,
    RealtimeAudioFormatsAudioPcm,
    VoiceAgentAudioConfig,
    VoiceAgentAudioInputConfig,
    VoiceAgentAudioOutputConfig,
    VoiceAgentDefinition,
    VoiceAgentEndConversationSystemTool,
    VoiceAgentInputTranscription,
    VoiceAgentInputTranscriptionModel,
    VoiceAgentServerVadTurnDetection,
    VoiceModelType,
    VoiceOutputModality,
    VoiceType,
)

VOICE_AGENT_NAME = "it-desk-voice-agent"
preview = AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=sync_cred, allow_preview=True)

def desk_definition(instructions: str) -> VoiceAgentDefinition:
    return VoiceAgentDefinition(
        model_type=VoiceModelType.MANAGED,
        model=VOICE_AGENT_MODEL,
        instructions=instructions,
        audio=VoiceAgentAudioConfig(
            input=VoiceAgentAudioInputConfig(
                format=RealtimeAudioFormatsAudioPcm(rate=24000),
                turn_detection=VoiceAgentServerVadTurnDetection(
                    threshold=0.5, prefix_padding_ms=300, silence_duration_ms=500),
                transcription=VoiceAgentInputTranscription(
                    model=VoiceAgentInputTranscriptionModel.WHISPER1),
            ),
            output=VoiceAgentAudioOutputConfig(
                voice="en-US-Ava:DragonHDLatestNeural", voice_type=VoiceType.AZURE_STANDARD),
        ),
        output_modalities=[VoiceOutputModality.AUDIO],
        tools=[VoiceAgentEndConversationSystemTool()],
        store=True,
    )

created_versions = []
v1 = preview.agents.create_version(agent_name=VOICE_AGENT_NAME,
                                   definition=desk_definition(DESK_INSTRUCTIONS))
created_versions.append(v1)
print("created version", v1.version)

v2 = preview.agents.create_version(
    agent_name=VOICE_AGENT_NAME,
    definition=desk_definition(DESK_INSTRUCTIONS + "\nEnd every answer by asking if there is anything else."),
)
created_versions.append(v2)
print("updated -> version", v2.version)

print("\nvoice agents in this project:")
for a in preview.agents.list(kind=AgentKind.VOICE):
    print(" -", a.name)

**Tools on a voice agent.** The definition above attaches **only** the `end_conversation` system
tool; there is no ticket tool. A voice agent's `tools` list accepts four kinds:

| Kind | Executed by | How you attach it |
|---|---|---|
| `toolbox` | The service | A versioned Foundry **toolbox** (`VoiceAgentToolboxTool`). This is how server-side tools — web search, Azure AI Search, OpenAPI — reach a voice agent; they can't be declared on it directly. |
| `mcp` | The service | One remote MCP server (`VoiceAgentMcpTool`), for a server not managed through a toolbox. |
| `function` | Your client | `VoiceAgentFunctionTool`: the service forwards the call over the live session and your client returns the result, the same round trip as step 5. |
| `system` | The service | Session controls such as `end_conversation`. |

In the portal playground, an agent carries **either one toolbox or individual tools, not both**;
attaching a toolbox replaces the individual list. The toolbox is the preferred route because it adds reuse, versioning,
governance and tool search. For the service desk, the ticketing lookup would sit behind an
OpenAPI or MCP tool in a toolbox, so the service calls it and no client code is involved.
See [Attach tools](https://learn.microsoft.com/azure/foundry/agents/how-to/configure-voice-agent#attach-tools)
and [toolboxes](https://learn.microsoft.com/azure/foundry/agents/how-to/tools/toolbox).

**An existing text agent can't be switched to voice.** Keep it, and attach it to a new voice agent
as a **subagent** (`subagent_config`). The subagent must be in the same project.

## 8 · Talk to the voice agent  *(deck slide 6, "connect by name")*

The client opens a real-time session **by agent name**. The SDK resolves the WebSocket address and
adds the preview header. The client sends **no** `session.update`, because the agent definition
already holds the model, instructions, voice and turn detection.

The conversation ID arrives on `session.created`. Keep it: step 9 uses it to read the conversation
back. The two typed turns below share one session, so the agent remembers the first turn when it
answers the second.

> These realtime classes come from the `azure-ai-projects` voice extra and follow the official sample
> `voice_agent_realtime_text_conversation_async.py`. Being preview, they are the most likely part of
> this notebook to change. If a name fails, compare it with that sample first.

In [ ]:
from azure.ai.projects.aio import AIProjectClient as AsyncProjectClient
from azure.ai.projects.models import (
    RealtimeConversationItemMessageUser,
    RealtimeConversationItemMessageUserContent,
    RealtimeServerEventError,
    RealtimeServerEventResponseAudioDelta,
    RealtimeServerEventResponseAudioTranscriptDone,
    RealtimeServerEventResponseDone,
    RealtimeServerEventSessionCreated,
)

async def agent_turn(conn, timeout=60):
    """Read typed voice-agent events until the response is done."""
    rec = {"audio": bytearray(), "transcript": "", "user_transcript": "",
           "first_audio_s": None, "conversation_id": None}
    t0 = time.perf_counter()
    async def _loop():
        async for ev in conn:
            if isinstance(ev, RealtimeServerEventSessionCreated):
                rec["conversation_id"] = ev.conversation_id
            elif isinstance(ev, RealtimeServerEventResponseAudioDelta):
                if rec["first_audio_s"] is None:
                    rec["first_audio_s"] = time.perf_counter() - t0
                rec["audio"] += ev.delta                 # already-decoded PCM16
            elif isinstance(ev, RealtimeServerEventResponseAudioTranscriptDone):
                rec["transcript"] += ev.transcript
            elif isinstance(ev, RealtimeServerEventError):
                raise RuntimeError(ev.error.message)
            elif isinstance(ev, RealtimeServerEventResponseDone):
                return
    await asyncio.wait_for(_loop(), timeout)
    return rec

conversation_id = None
async with AsyncProjectClient(endpoint=PROJECT_ENDPOINT, credential=credential, allow_preview=True) as aproject:
    async with aproject.beta.voice_agents.realtime.connect(agent_name=VOICE_AGENT_NAME) as conn:
        for question in ["Hi, I forgot my password. What should I do?",
                         "Thanks. Can I also get a new monitor through this line?"]:
            await conn.conversation.item.create(item=RealtimeConversationItemMessageUser(
                content=[RealtimeConversationItemMessageUserContent(type="input_text", text=question)]))
            await conn.response.create()
            turn = await agent_turn(conn)
            conversation_id = conversation_id or turn["conversation_id"]
            print("caller :", question)
            play(turn, "agent")

print("conversation id:", conversation_id)

**Speaking first.** A voice agent can open the call itself: set a `greeting` in the definition, and
the service plays it as soon as the session opens, before any caller turn. A client must then read
that greeting response before sending its first message. The official sample shows how. This
notebook leaves the greeting out so each turn is easy to follow.

## 9 · Read the conversation back, then clean up  *(deck slide 8, "voice tracing")*

Because the definition set `store=True`, Foundry kept the conversation: the **ordered transcript
items**, the responses with **token usage**, and the **audio**. This is what call review, quality
checks and dispute handling need. With path 2 (step 6), this record would have to be built by the
application team.

Recording is **off by default**. Turn it on only with a clear retention and consent policy; a
recording of a caller is personal data.

In [ ]:
convs = preview.beta.voice_agents.conversations

conv = convs.get(VOICE_AGENT_NAME, conversation_id)
print(f"conversation {conv.id}: status={conv.status}, created_at={conv.created_at}\n")

print("transcript items:")
for item in convs.list_items(VOICE_AGENT_NAME, conversation_id):
    print(f"  - {item.type:28s} id={getattr(item, 'id', None)}")

In [ ]:
# Clean up the versions this notebook created. Deleting the last version removes the agent.
for v in reversed(created_versions):
    preview.agents.delete_version(agent_name=VOICE_AGENT_NAME, agent_version=v.version)
    print("deleted voice agent version", v.version)

# Deleting a conversation removes its responses, items and audio. Uncomment to run:
# convs.delete(VOICE_AGENT_NAME, conversation_id)

await credential.close()

---
## What you built, and what comes next

| Step | Capability | Path on deck slide 3 | Status |
|---|---|---|---|
| 1 | Session, proactive greeting, streamed audio | 1 · Voice Live with a model | generally available |
| 2 | HD and standard voices | 1 | generally available |
| 3 | Audio streaming, turn detection, noise and echo control, transcription | 1 | generally available |
| 4 | Single speech model vs three stages | 1 | generally available |
| 5 | Function tools against a back-end system | 1 | generally available |
| 6 | Voice in front of an existing Foundry agent | 2 · Voice Live with a Foundry agent | generally available |
| 7–9 | Versioned voice agent, live session, stored conversation | 3 · Foundry voice agent | **preview** |

**Not covered here, on purpose:**

- **Phone numbers.** Voice agents connect to Teams Phone or Twilio, inbound and outbound, and the
  setup is in the portal. See
  [telephony channels](https://learn.microsoft.com/en-us/azure/foundry/agents/how-to/voice-agent-telephony-channels).
- **Browser clients and avatars** use WebRTC (preview), which needs a web page rather than a
  notebook. See the avatar and universal assistant samples in
  [voicelive-samples](https://github.com/microsoft-foundry/voicelive-samples).
- **Your own voice framework.** Running Pipecat, LiveKit or Voice Live inside a hosted agent
  container uses the Invocations WebSocket protocol (deck slide 9).

**Limits to keep in mind** (Standard tier, per resource): 100 new connections per minute, sessions of
up to 60 minutes, and up to 120,000 tokens per minute. Billing is per token, split into text, audio
and native audio; the model sets the pricing tier. Check the
[quotas page](https://learn.microsoft.com/en-us/azure/ai-services/speech-service/speech-services-quotas-and-limits)
and the [pricing page](https://azure.microsoft.com/pricing/details/speech/) before sizing anything.

**Before you build on any of this:** preview status, model lists and regions change frequently.
Re-check the Learn pages above rather than relying on a notebook dated October 2026.